# Scikit-learn 분류 (Classification)

다양한 분류 알고리즘을 학습하고 비교합니다.

## 학습 목표
1. 분류 알고리즘 이해
2. 모델 학습 및 평가
3. 교차 검증
4. 하이퍼파라미터 튜닝

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

## 1. 데이터 로드 및 탐색

In [ ]:
# Iris 데이터셋 로드
iris = load_iris()
X, y = iris.data, iris.target

print(f"특성 shape: {X.shape}")
print(f"타겟 shape: {y.shape}")
print(f"특성 이름: {iris.feature_names}")
print(f"클래스: {iris.target_names}")

In [ ]:
# 데이터 시각화
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, feature in zip(axes.flat, iris.feature_names):
    for species in df['species'].unique():
        subset = df[df['species'] == species]
        ax.hist(subset[feature], alpha=0.5, label=species, bins=15)
    ax.set_xlabel(feature)
    ax.legend()
plt.tight_layout()
plt.show()

## 2. 데이터 분할 및 전처리

In [ ]:
# 학습/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"학습 데이터: {X_train.shape}")
print(f"테스트 데이터: {X_test.shape}")

## 3. 다양한 분류 모델 학습 및 비교

In [ ]:
# 모델 정의
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(),
    'SVM': SVC(kernel='rbf'),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

# 모델 학습 및 평가
results = []
for name, model in models.items():
    # 학습
    model.fit(X_train_scaled, y_train)
    
    # 예측
    y_pred = model.predict(X_test_scaled)
    
    # 교차 검증
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    
    # 결과 저장
    results.append({
        'Model': name,
        'Test Accuracy': accuracy_score(y_test, y_pred),
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std()
    })

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
# 결과 시각화
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results_df))
width = 0.35

ax.bar(x - width/2, results_df['Test Accuracy'], width, label='Test Accuracy')
ax.bar(x + width/2, results_df['CV Mean'], width, label='CV Mean', yerr=results_df['CV Std'])

ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0.8, 1.05)

plt.tight_layout()
plt.show()

## 4. 하이퍼파라미터 튜닝

In [ ]:
# Random Forest 하이퍼파라미터 튜닝
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print(f"최적 파라미터: {grid_search.best_params_}")
print(f"최적 CV 점수: {grid_search.best_score_:.4f}")

# 최적 모델로 테스트
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)
print(f"테스트 정확도: {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# 혼동 행렬
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# 분류 리포트
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## 5. 특성 중요도

In [ ]:
# Random Forest 특성 중요도
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.bar(range(X.shape[1]), importances[indices])
plt.xticks(range(X.shape[1]), [iris.feature_names[i] for i in indices], rotation=45)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

## 연습 문제

1. Wine 데이터셋으로 같은 분석을 수행해보세요.
2. 더 많은 하이퍼파라미터를 튜닝해보세요.
3. XGBoost, LightGBM을 추가로 비교해보세요.